# Replication of the GTCN-G paper setup on UNSW-NB15 (baseline: E-GraphSAGE-M)

GTCN-G (Xu et al., arXiv 2510.07285) reports weighted F1 on 10-class UNSW-NB15 with a 5:2:3 train/validation/test split, 10 epochs and mini-batches of 500. Its stated learning rate of 0.007 is for GTCN-G; the baselines' rate is not given. Its baselines are E-GraphSAGE (0.8756), E-GraphSAGE-M (0.8934), GAT (0.9178) and GTCN-G itself (0.9512). The paper has no code.

This notebook builds the closest version of that setup we can, using their baseline E-GraphSAGE-M: a two-layer mean-aggregator E-GraphSAGE, 8 sampled neighbours per hop, batches of 500 flows, learning rate 0.001 (the original E-GraphSAGE value), plain cross-entropy. A first run with 0.007 collapsed to predicting Normal for every flow within two epochs in all three seeds, so it was not used.

What the paper does not state, and what we chose:
- Split: stratified random 50% / 20% / 30%.
- Data: our deduplicated, preprocessed UNSW-NB15 (train and test files of the project joined, then re-split).
- Loss, optimizer, hidden size: plain cross-entropy, Adam, 128 units.

We train well past their 10 epochs, halving the learning rate whenever validation weighted F1 stalls for 5 epochs so that the curves settle, with early stopping on the same metric. We report the test scores both after epoch 10 and at the best validation epoch. The test set is never used for training or stopping.

In [ ]:
import copy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

DEBUG = False
SEEDS = [42] if DEBUG else [42, 43, 44]
PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 128
DROPOUT = 0.2
LR = 0.001
BATCH = 500
FANOUT = 8
PAPER_EPOCHS = 10
MAX_EPOCHS = 12 if DEBUG else 100
PATIENCE = 12 if DEBUG else 20
PAPER = {"E-GraphSAGE": 0.8756, "E-GraphSAGE-M": 0.8934, "GAT": 0.9178, "GTCN-G": 0.9512}

## 1. Data and 5:2:3 split

In [ ]:
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
graph = pd.concat([pd.read_csv(f"{PATH}/train_graph.csv", index_col="row_id"),
                   pd.read_csv(f"{PATH}/test_graph.csv", index_col="row_id")]).loc[df.index]
if DEBUG:
    df = df.sample(200_000, random_state=0)
    graph = graph.loc[df.index]

class_names = list(joblib.load(f"{PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
feature_cols = [c for c in df.columns if c != "label"]

train_ids, rest_ids = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
val_ids, test_ids = train_test_split(rest_ids, train_size=0.4, stratify=df.loc[rest_ids, "label"], random_state=0)
split = pd.Series("test", index=df.index)
split.loc[train_ids], split.loc[val_ids] = "train", "val"
split.to_csv("/kaggle/working/split.csv")
print({k: len(v) for k, v in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}, "| features", len(feature_cols))

## 2. Graphs

Nodes are (IP, port) endpoints and flows are edges. Each split gets its own graph built from its own flows. A graph is stored as a neighbour list (CSR) so that neighbours can be sampled on the GPU.

In [ ]:
def build_graph(ids):
    feat_df, graph_df = df.loc[ids], graph.loc[ids]
    n = len(feat_df)
    ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
    ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
    node_id, uniq = pd.factorize(ip_code.astype("int64") * 70_000 + ports)
    node_id = torch.from_numpy(node_id).to(DEVICE)
    src, dst = node_id[:n], node_id[n:]

    nodes = torch.cat([src, dst])
    other = torch.cat([dst, src])
    flow = torch.cat([torch.arange(n, device=DEVICE)] * 2)
    order = torch.argsort(nodes)
    ptr = torch.zeros(len(uniq) + 1, dtype=torch.long, device=DEVICE)
    ptr[1:] = torch.cumsum(torch.bincount(nodes, minlength=len(uniq)), 0)
    return {"src": src, "dst": dst, "nbr": other[order], "eflow": flow[order], "ptr": ptr,
            "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")).to(DEVICE),
            "y": torch.from_numpy(feat_df["label"].to_numpy("int64")).to(DEVICE)}


G = {"train": build_graph(train_ids), "val": build_graph(val_ids), "test": build_graph(test_ids)}
for name, g in G.items():
    deg = torch.diff(g["ptr"]).float()
    print(f"{name:<6} edges {len(g['src']):>9,} | nodes {len(deg):>9,} | mean degree {deg.mean():.2f} | single-flow nodes {(deg == 1).float().mean():.1%}")

## 3. Model

For a batch of flows, the two endpoints are embedded with two layers of sampled neighbourhood aggregation (8 neighbours per hop). The message from a neighbour is a linear function of its embedding and the connecting flow's features; the node update is a linear layer on [own embedding, mean message]. The node feature is a constant. The classifier is one linear layer on the concatenated source and destination embeddings.

In [ ]:
def sample(g, nodes):
    start = g["ptr"][nodes]
    deg = g["ptr"][nodes + 1] - start
    pos = start[:, None] + (torch.rand(len(nodes), FANOUT, device=DEVICE) * deg[:, None]).long()
    return g["nbr"][pos], g["eflow"][pos]


class EGraphSAGEM(nn.Module):
    def __init__(self, n_feat, n_classes):
        super().__init__()
        self.neigh = nn.ModuleList([nn.Linear(1, HIDDEN, bias=False), nn.Linear(HIDDEN, HIDDEN, bias=False)])
        self.edge = nn.ModuleList([nn.Linear(n_feat, HIDDEN), nn.Linear(n_feat, HIDDEN)])
        self.update = nn.ModuleList([nn.Linear(1 + HIDDEN, HIDDEN), nn.Linear(2 * HIDDEN, HIDDEN)])
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(2 * HIDDEN, n_classes)

    def forward(self, g, idx):
        ends = torch.cat([g["src"][idx], g["dst"][idx]])
        n1, e1 = sample(g, ends)
        hop1 = torch.cat([ends, n1.flatten()])
        _, e2 = sample(g, hop1)

        ones = torch.ones(len(hop1), 1, device=DEVICE)
        msg = self.neigh[0](ones) + self.edge[0](g["x"][e2].mean(1))
        h1 = self.drop(torch.relu(self.update[0](torch.cat([ones, msg], dim=1))))

        h_ends, h_nbr = h1[:len(ends)], h1[len(ends):].view(len(ends), FANOUT, HIDDEN)
        msg = self.neigh[1](h_nbr.mean(1)) + self.edge[1](g["x"][e1].mean(1))
        h2 = torch.relu(self.update[1](torch.cat([h_ends, msg], dim=1)))
        return self.out(torch.cat([h2[:len(idx)], h2[len(idx):]], dim=1))


loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(model, g, batch=5000):
    model.eval()
    logits = torch.cat([model(g, torch.arange(i, min(i + batch, len(g["src"])), device=DEVICE))
                        for i in range(0, len(g["src"]), batch)])
    return loss_fn(logits, g["y"]).item(), logits.argmax(1).cpu().numpy()


def scores(y_true, pred):
    per_class = f1_score(y_true, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    return {"weighted F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "macro F1": per_class.mean(), "accuracy": accuracy_score(y_true, pred)}, per_class

## 4. Training

Each epoch is one pass over the training flows in batches of 500. Early stopping watches validation weighted F1 and waits 20 epochs for an improvement; the learning rate is halved after 5 epochs without one.

In [ ]:
def run(seed):
    torch.manual_seed(seed)
    model = EGraphSAGEM(len(feature_cols), n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    g_train, y_val, y_test = G["train"], G["val"]["y"].cpu().numpy(), G["test"]["y"].cpu().numpy()
    n_train = len(g_train["src"])

    history, best, bad, at_paper = [], {"f1": -1.0}, 0, None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        total = torch.zeros((), device=DEVICE)
        for i in range(0, n_train, BATCH):
            idx = perm[i:i + BATCH]
            loss = loss_fn(model(g_train, idx), g_train["y"][idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(idx)

        val_loss, val_pred = evaluate(model, G["val"])
        val, _ = scores(y_val, val_pred)
        history.append({"seed": seed, "epoch": epoch, "train loss": total.item() / n_train, "val loss": val_loss,
                        "val weighted F1": val["weighted F1"], "val macro F1": val["macro F1"]})
        print(f"seed {seed} epoch {epoch:>3} | train loss {history[-1]['train loss']:.4f} | val loss {val_loss:.4f} "
              f"| val weighted F1 {val['weighted F1']:.4f} | val macro F1 {val['macro F1']:.4f}", flush=True)

        sched.step(val["weighted F1"])
        if epoch == PAPER_EPOCHS:
            at_paper, _ = scores(y_test, evaluate(model, G["test"])[1])
        if val["weighted F1"] > best["f1"]:
            best, bad = {"f1": val["weighted F1"], "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    test_pred = evaluate(model, G["test"])[1]
    at_best, per_class = scores(y_test, test_pred)
    row = {"seed": seed, "best epoch": best["epoch"], "last epoch": epoch,
           **{f"{k} (epoch {PAPER_EPOCHS})": v for k, v in at_paper.items()},
           **{f"{k} (best epoch)": v for k, v in at_best.items()}}
    return pd.DataFrame(history), row, {"seed": seed, **dict(zip(class_names, per_class))}, test_pred.astype("int8")


histories, rows, class_rows, preds = [], [], [], {}
for seed in SEEDS:
    history, row, class_row, pred = run(seed)
    histories.append(history)
    rows.append(row)
    class_rows.append(class_row)
    preds[f"seed_{seed}"] = pred
    pd.concat(histories).to_csv("/kaggle/working/paper_split_history.csv", index=False)
    pd.DataFrame(rows).to_csv("/kaggle/working/paper_split_runs.csv", index=False)
    pd.DataFrame(class_rows).to_csv("/kaggle/working/paper_split_per_class.csv", index=False)
history = pd.concat(histories)
results = pd.DataFrame(rows)
class_results = pd.DataFrame(class_rows)
pd.DataFrame(preds, index=test_ids).to_parquet("/kaggle/working/paper_split_test_pred.parquet")

## 5. Training curves

Dashed line: epoch 10, where the paper stops.

In [ ]:
def curves(columns, titles, path):
    fig, axes = plt.subplots(1, len(columns), figsize=(7 * len(columns), 4.5))
    for ax, col, title in zip(np.atleast_1d(axes), columns, titles):
        sns.lineplot(data=history, x="epoch", y=col, hue="seed", palette="deep", ax=ax)
        ax.axvline(PAPER_EPOCHS, color="gray", linestyle="--")
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()


curves(["train loss", "val loss"], ["Training loss", "Validation loss"], "/kaggle/working/fig_01.png")
curves(["val weighted F1", "val macro F1"], ["Validation weighted F1", "Validation macro F1"], "/kaggle/working/fig_02.png")

## 6. Test results

In [ ]:
summary = results.drop(columns=["seed", "best epoch", "last epoch"]).agg(["mean", "std"]).T
print(summary.round(4).to_string())
print("best epochs:", results["best epoch"].tolist(), "| last epochs:", results["last epoch"].tolist())
summary.to_csv("/kaggle/working/paper_split_summary.csv")

bars = {**PAPER, f"ours, epoch {PAPER_EPOCHS}": results[f"weighted F1 (epoch {PAPER_EPOCHS})"].mean(),
        "ours, best epoch": results["weighted F1 (best epoch)"].mean()}
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(list(bars), list(bars.values()), color=["gray"] * len(PAPER) + ["steelblue"] * 2)
ax.set_xlim(0.8, 1.0)
ax.set_title("Test weighted F1: paper (reported) and ours (mean over seeds)")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_03.png", dpi=130, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=class_results.melt(id_vars="seed", var_name="class", value_name="F1"), x="class", y="F1", errorbar="sd", color="steelblue", ax=ax)
ax.set_title("Test F1 per class at the best validation epoch (mean and std over seeds)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("/kaggle/working/fig_04.png", dpi=130, bbox_inches="tight")
plt.show()